In [2]:
!pip install emnist tensorflow seaborn -q

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from emnist import extract_training_samples
from emnist import extract_test_samples

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
from torchvision import datasets

# Download and load the 'letters' split
train_data = datasets.EMNIST(root='./data', split='letters', train=True, download=True)
test_data  = datasets.EMNIST(root='./data', split='letters', train=False, download=True)

# Extract as standard Numpy arrays
X_train, y_train = train_data.data.numpy(), train_data.targets.numpy()
X_test, y_test   = test_data.data.numpy(), test_data.targets.numpy()

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

In [ ]:
plt.figure(figsize=(12,6))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(X_train[i], cmap='gray')

    plt.title(chr(y_train[i]+64))

    plt.axis('off')

plt.tight_layout()
plt.show()

In [7]:
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

y_train = y_train - 1
y_test = y_test - 1

y_train_cat = to_categorical(y_train,26)
y_test_cat = to_categorical(y_test,26)

In [ ]:
model = Sequential()

model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(28,28,1)
    )
)

model.add(
    MaxPooling2D((2,2))
)

model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

model.add(
    MaxPooling2D((2,2))
)

model.add(Flatten())

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(
    Dropout(0.3)
)

model.add(
    Dense(
        26,
        activation='softmax'
    )
)

model.summary()

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=10,
    batch_size=128
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Accuracy Curve")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(
    ["Training","Validation"]
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Loss Curve")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend(
    ["Training","Validation"]
)

plt.show()

In [13]:
loss, accuracy = model.evaluate(
    X_test,
    y_test_cat
)

print(
    "Test Accuracy:",
    round(accuracy*100,2),
    "%"
)

650/650 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9360 - loss: 0.2050
Test Accuracy: 93.6 %


In [ ]:
predictions = model.predict(X_test)

predicted_labels = np.argmax(
    predictions,
    axis=1
)

In [ ]:
cm = confusion_matrix(
    y_test,
    predicted_labels
)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
print(
    classification_report(
        y_test,
        predicted_labels
    )
)

In [ ]:
plt.figure(figsize=(12,8))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(
        X_test[i].reshape(28,28),
        cmap='gray'
    )

    actual = chr(y_test[i]+65)
    pred = chr(predicted_labels[i]+65)

    plt.title(
        f"A:{actual} P:{pred}"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model.save(
    "handwritten_character_model.h5"
)

print("Model Saved Successfully")